In [ ]:

# ============================================================
# HYPOTHESIS 3: MONTE CARLO POWER ANALYSIS WITH LCA
# HBCD STUDY RELEASE 2.1
# ============================================================
#
# Install if needed:
# pip install numpy pandas scipy statsmodels joblib
# pip install threadpoolctl tqdm matplotlib
#
# Text-based progress bar: no Jupyter widgets required.
# ============================================================

import os
import sys
import warnings
import json
from pathlib import Path
from itertools import product

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import logsumexp
from scipy.optimize import linear_sum_assignment
from scipy.stats import chi2, truncnorm

from statsmodels.regression.mixed_linear_model import MixedLM
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits
from tqdm import tqdm


# ============================================================
# 1. SETTINGS
# ============================================================

N_GM = 1500
N_WM = 1235

N_STATES = 20
N_SITES = 27

N_SIMULATIONS = 500

N_JOBS = 4
BATCH_SIZE = 4

SEED = 12345

ALPHA = 0.05
TARGET_POWER = 0.95

EFFECT_SIZES = [
    0.00, 0.05, 0.10, 0.15, 0.20,
    0.25, 0.30, 0.35, 0.40, 0.50
]

PREDICTORS = [
    "Income-to-needs ratio",
    "Maternal substance use",
    "Maternal depression"
]

OUTCOMES = ["GM", "WM"]

GENERATING_CLASSES = [2, 3]
CANDIDATE_CLASSES = [1, 2, 3, 4, 5]

LCA_N_STARTS = 10
LCA_MAX_ITER = 300
LCA_TOL = 1e-6

MIN_CLASS_PROPORTION = 0.05

VAR_STATE = 0.03
VAR_SITE = 0.04
VAR_FAMILY = 0.03
VAR_RESIDUAL = 0.90

BETA_PREDICTOR = 0.10

GM_MEAN = 1220
GM_SD = 187

THREE_CLASS_EFFECT_PATTERN = np.array([
    0.0, 1.0, 1.0
])

OUTPUT_DIR = Path(
    f"HBCD_H3_LCA_power_{N_SIMULATIONS}sim_text"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. RESOURCE RECEIPT INDICATORS
# ============================================================

RESOURCE_NAMES = [
    "SNAP",
    "WIC",
    "Housing",
    "Cash",
    "Unemployment",
    "Childcare"
]

RESOURCE_PREVALENCE = np.array([
    0.2832,
    0.2586,
    0.0533,
    0.0387,
    0.0193,
    0.0434
])

CLASS_PROPORTIONS = {
    2: np.array([0.70, 0.30]),
    3: np.array([0.55, 0.35, 0.10])
}

RELATIVE_ITEM_WEIGHTS = {

    2: np.array([
        [0.40, 0.40, 0.45, 0.45, 0.45, 0.45],
        [2.40, 2.40, 2.20, 2.20, 2.20, 2.20]
    ]),

    3: np.array([
        [0.35, 0.35, 0.35, 0.35, 0.35, 0.35],
        [2.10, 2.10, 0.65, 0.65, 0.65, 0.65],
        [2.50, 2.50, 5.00, 5.00, 5.00, 5.00]
    ])
}


def calibrate_item_probabilities(k):

    weights = RELATIVE_ITEM_WEIGHTS[k]
    proportions = CLASS_PROPORTIONS[k]

    denominator = (
        proportions[:, None] * weights
    ).sum(axis=0)

    probabilities = (
        RESOURCE_PREVALENCE[None, :]
        * weights
        / denominator[None, :]
    )

    if np.any(probabilities <= 0) or np.any(probabilities >= 1):
        raise ValueError(
            "Invalid class-specific probabilities."
        )

    return probabilities


ITEM_PROBABILITIES = {
    k: calibrate_item_probabilities(k)
    for k in GENERATING_CLASSES
}

for k in GENERATING_CLASSES:

    print(f"\n{k}-CLASS ITEM PROBABILITIES")

    print(
        pd.DataFrame(
            ITEM_PROBABILITIES[k],
            columns=RESOURCE_NAMES
        ).round(3)
    )


# ============================================================
# 3. STANDARDIZATION
# ============================================================

def standardize(x):

    x = np.asarray(x, dtype=float)

    sd = x.std(ddof=0)

    if sd <= 0:
        raise ValueError(
            "Cannot standardize constant variable."
        )

    return (x - x.mean()) / sd


# ============================================================
# 4. SIMULATE FAMILY CHARACTERISTICS
# ============================================================

def simulate_predictor(n, predictor, rng):

    if predictor == "Income-to-needs ratio":

        x = rng.lognormal(
            mean=0,
            sigma=0.75,
            size=n
        )

    elif predictor == "Maternal substance use":

        mean = 4.0
        variance = 2.88 ** 2

        r = mean ** 2 / (variance - mean)
        p = r / (r + mean)

        x = rng.negative_binomial(
            r, p, size=n
        )

        x = np.clip(x, 0, 19)

    elif predictor == "Maternal depression":

        mean = 46.33
        sd = 8.12

        lower = (38.20 - mean) / sd
        upper = (81.30 - mean) / sd

        x = truncnorm.rvs(
            lower,
            upper,
            loc=mean,
            scale=sd,
            size=n,
            random_state=rng
        )

    else:
        raise ValueError("Unknown predictor.")

    return standardize(x)


# ============================================================
# 5. SIMULATE MULTILEVEL STRUCTURE
# ============================================================

def simulate_structure(n, rng):

    state_idx = np.tile(
        np.arange(N_STATES),
        int(np.ceil(n / N_STATES))
    )[:n]

    rng.shuffle(state_idx)

    site_counts = np.array(
        [2] * 7 + [1] * 13
    )

    site_starts = np.cumsum(
        np.r_[0, site_counts[:-1]]
    )

    site_idx = np.empty(n, dtype=int)

    for state in range(N_STATES):

        members = np.where(
            state_idx == state
        )[0]

        site_idx[members] = (
            site_starts[state]
            + rng.integers(
                0,
                site_counts[state],
                size=len(members)
            )
        )

    family_idx = np.arange(n)

    n_pairs = int(
        round(0.05 * n / 2)
    )

    available = {}

    for state in range(N_STATES):

        members = list(
            np.where(state_idx == state)[0]
        )

        rng.shuffle(members)
        available[state] = members

    for _ in range(n_pairs):

        eligible = [
            s for s in range(N_STATES)
            if len(available[s]) >= 2
        ]

        if not eligible:
            break

        state = int(
            rng.choice(eligible)
        )

        a = available[state].pop()
        b = available[state].pop()

        family_idx[b] = family_idx[a]

    df = pd.DataFrame({
        "state_idx": state_idx,
        "site_idx": site_idx,
        "family_idx": family_idx
    })

    df["gestational_age"] = standardize(
        rng.normal(39, 1.5, n)
    )

    df["scan_age"] = standardize(
        rng.uniform(0, 12, n)
    )

    df["sex"] = rng.binomial(
        1, 0.5, n
    )

    df["maternal_education"] = standardize(
        rng.normal(15, 2.5, n)
    )

    df["ICV"] = standardize(
        rng.normal(0, 1, n)
    )

    df["scanner_manufacturer"] = rng.choice(
        ["GE", "Siemens", "Philips"],
        size=n,
        p=[0.25, 0.60, 0.15]
    )

    df["race_ethnicity"] = rng.choice(
        ["Group1", "Group2", "Group3", "Group4"],
        size=n,
        p=[0.45, 0.25, 0.20, 0.10]
    )

    return df


# ============================================================
# 6. GENERATE LATENT CLASSES AND RESOURCE INDICATORS
# ============================================================

def simulate_resources(n, k, rng):

    true_class = rng.choice(
        k,
        size=n,
        p=CLASS_PROPORTIONS[k]
    )

    probabilities = (
        ITEM_PROBABILITIES[k][true_class]
    )

    indicators = rng.binomial(
        1, probabilities
    ).astype(float)

    return true_class, indicators


# ============================================================
# 7. BINARY LCA VIA EM
# ============================================================

def fit_binary_lca(
    X,
    k,
    rng,
    n_starts=LCA_N_STARTS,
    max_iter=LCA_MAX_ITER,
    tol=LCA_TOL
):

    X = np.asarray(X, dtype=float)

    n, n_items = X.shape
    best = None

    for start in range(n_starts):

        pi = rng.dirichlet(
            np.ones(k)
        )

        theta = rng.uniform(
            0.05,
            0.80,
            size=(k, n_items)
        )

        previous_ll = -np.inf
        converged = False

        for iteration in range(max_iter):

            log_prob = (
                np.log(pi)[None, :]
                + X @ np.log(theta).T
                + (1 - X) @ np.log(1 - theta).T
            )

            log_denom = logsumexp(
                log_prob,
                axis=1,
                keepdims=True
            )

            posterior = np.exp(
                log_prob - log_denom
            )

            ll = float(
                log_denom.sum()
            )

            nk = posterior.sum(axis=0)

            pi_new = nk / n

            theta_new = (
                posterior.T @ X
            ) / np.maximum(
                nk[:, None], 1e-10
            )

            pi_new = np.clip(
                pi_new, 1e-8, None
            )

            pi_new /= pi_new.sum()

            theta_new = np.clip(
                theta_new,
                1e-6,
                1 - 1e-6
            )

            pi = pi_new
            theta = theta_new

            if abs(ll - previous_ll) < tol:
                converged = True
                break

            previous_ll = ll

        log_prob = (
            np.log(pi)[None, :]
            + X @ np.log(theta).T
            + (1 - X) @ np.log(1 - theta).T
        )

        log_denom = logsumexp(
            log_prob,
            axis=1,
            keepdims=True
        )

        posterior = np.exp(
            log_prob - log_denom
        )

        ll = float(
            log_denom.sum()
        )

        n_parameters = (
            (k - 1) + k * n_items
        )

        aic = (
            -2 * ll + 2 * n_parameters
        )

        bic = (
            -2 * ll
            + np.log(n) * n_parameters
        )

        if k > 1:

            raw_entropy = -np.sum(
                posterior * np.log(
                    np.clip(
                        posterior, 1e-12, 1
                    )
                )
            )

            entropy = (
                1
                - raw_entropy
                / (n * np.log(k))
            )

        else:
            entropy = np.nan

        assigned = np.argmax(
            posterior,
            axis=1
        )

        class_sizes = np.bincount(
            assigned,
            minlength=k
        )

        class_proportions = (
            class_sizes / n
        )

        candidate = {
            "pi": pi.copy(),
            "theta": theta.copy(),
            "posterior": posterior.copy(),
            "loglik": ll,
            "aic": aic,
            "bic": bic,
            "entropy": entropy,
            "converged": converged,
            "iterations": iteration + 1,
            "class_sizes": class_sizes,
            "class_proportions": class_proportions,
            "minimum_class_proportion": (
                class_proportions.min()
            ),
            "average_posterior_probability": (
                posterior.max(axis=1).mean()
            )
        }

        if (
            best is None
            or (
                converged
                and not best["converged"]
            )
            or (
                converged == best["converged"]
                and ll > best["loglik"]
            )
        ):
            best = candidate

    return best


# ============================================================
# 8. COMPARE 1–5 LCA CLASS SOLUTIONS
# ============================================================

def evaluate_lca_models(X, rng):

    models = {}
    rows = []

    for k in CANDIDATE_CLASSES:

        fit = fit_binary_lca(
            X, k, rng
        )

        models[k] = fit

        rows.append({
            "candidate_classes": k,
            "aic": fit["aic"],
            "bic": fit["bic"],
            "entropy": fit["entropy"],
            "converged": fit["converged"],
            "minimum_class_proportion": (
                fit["minimum_class_proportion"]
            ),
            "average_posterior_probability": (
                fit["average_posterior_probability"]
            )
        })

    comparison = pd.DataFrame(rows)

    eligible = comparison.loc[
        comparison["converged"]
    ]

    if len(eligible) == 0:

        selected_aic = np.nan
        selected_bic = np.nan

    else:

        selected_aic = int(
            eligible.loc[
                eligible["aic"].idxmin(),
                "candidate_classes"
            ]
        )

        selected_bic = int(
            eligible.loc[
                eligible["bic"].idxmin(),
                "candidate_classes"
            ]
        )

    return {
        "models": models,
        "comparison": comparison,
        "selected_aic": selected_aic,
        "selected_bic": selected_bic
    }


# ============================================================
# 9. ALIGN ESTIMATED CLASSES
# ============================================================

def align_classes(
    estimated_theta,
    reference_theta,
    posterior
):

    cost = (
        (
            estimated_theta[:, None, :]
            - reference_theta[None, :, :]
        ) ** 2
    ).sum(axis=2)

    estimated_idx, reference_idx = (
        linear_sum_assignment(cost)
    )

    aligned_posterior = np.zeros_like(
        posterior
    )

    for e, r in zip(
        estimated_idx, reference_idx
    ):

        aligned_posterior[:, r] = (
            posterior[:, e]
        )

    assigned = np.argmax(
        aligned_posterior,
        axis=1
    )

    return assigned, aligned_posterior


# ============================================================
# 10. GENERATE BRAIN OUTCOME
# ============================================================

def generate_outcome(
    df,
    predictor,
    true_class,
    k,
    beta,
    outcome_type,
    rng
):

    n = len(df)

    if k == 2:

        class_main = (
            true_class == 1
        ).astype(float)

        interaction = (
            beta * predictor * class_main
        )

    else:

        class_main = (
            true_class > 0
        ).astype(float)

        interaction = (
            beta
            * predictor
            * THREE_CLASS_EFFECT_PATTERN[true_class]
        )

    state_effects = rng.normal(
        0,
        np.sqrt(VAR_STATE),
        N_STATES
    )

    site_effects = rng.normal(
        0,
        np.sqrt(VAR_SITE),
        N_SITES
    )

    family_codes, family_labels = (
        pd.factorize(
            df["family_idx"],
            sort=False
        )
    )

    family_effects = rng.normal(
        0,
        np.sqrt(VAR_FAMILY),
        len(family_labels)
    )

    residual = rng.normal(
        0,
        np.sqrt(VAR_RESIDUAL),
        n
    )

    y = (
        BETA_PREDICTOR * predictor
        + 0.05 * class_main
        + interaction

        + 0.08 * df["gestational_age"].to_numpy()
        + 0.10 * df["scan_age"].to_numpy()
        + 0.05 * df["sex"].to_numpy()
        + 0.05 * df["maternal_education"].to_numpy()

        + state_effects[
            df["state_idx"].to_numpy()
        ]

        + site_effects[
            df["site_idx"].to_numpy()
        ]

        + family_effects[family_codes]

        + residual
    )

    if outcome_type == "GM":

        y += (
            0.25 * df["ICV"].to_numpy()
        )

        y = GM_MEAN + GM_SD * y

    return y


# ============================================================
# 11. FIT LINEAR MIXED-EFFECTS MODEL
# ============================================================

def fit_lme(df, outcome_type, k):

    formula = (
        "outcome ~ predictor * C(estimated_class)"
        " + gestational_age"
        " + scan_age"
        " + sex"
        " + maternal_education"
        " + C(scanner_manufacturer)"
        " + C(race_ethnicity)"
    )

    if outcome_type == "GM":
        formula += " + ICV"

    try:

        model = MixedLM.from_formula(
            formula=formula,
            data=df,
            groups="state_idx",
            re_formula="1",
            vc_formula={
                "site": "0 + C(site_idx)",
                "family": "0 + C(family_idx)"
            }
        )

    except Exception:

        return {
            "p_value": np.nan,
            "converged": False,
            "optimizer": "failed"
        }

    for method in ["lbfgs", "powell"]:

        try:

            with warnings.catch_warnings():

                warnings.simplefilter(
                    "ignore"
                )

                fit = model.fit(
                    reml=False,
                    method=method,
                    maxiter=500,
                    disp=False
                )

            if not fit.converged:
                continue

            names = [
                name
                for name in fit.fe_params.index
                if (
                    "predictor:C(estimated_class)"
                    in name
                )
            ]

            if len(names) != k - 1:
                continue

            coefficients = (
                fit.fe_params.loc[names]
                .to_numpy(dtype=float)
            )

            covariance = (
                fit.cov_params()
                .loc[names, names]
                .to_numpy(dtype=float)
            )

            statistic = float(
                coefficients
                @ np.linalg.solve(
                    covariance,
                    coefficients
                )
            )

            p_value = float(
                chi2.sf(
                    statistic,
                    df=k - 1
                )
            )

            return {
                "p_value": p_value,
                "converged": True,
                "optimizer": method,
                "wald_statistic": statistic
            }

        except Exception:
            continue

    return {
        "p_value": np.nan,
        "converged": False,
        "optimizer": "failed"
    }


# ============================================================
# 12. RUN ONE MONTE CARLO REPLICATE
# ============================================================

def run_one_replicate(
    scenario,
    replicate,
    seed
):

    outcome_type, predictor_name, k, beta = scenario

    rng = np.random.default_rng(seed)

    n = (
        N_GM if outcome_type == "GM"
        else N_WM
    )

    result = {
        "outcome_type": outcome_type,
        "predictor": predictor_name,
        "n_classes": k,
        "beta": beta,
        "replicate": replicate,
        "p_value": np.nan,
        "converged": False,
        "lca_converged": False,
        "error": ""
    }

    try:

        df = simulate_structure(
            n, rng
        )

        x = simulate_predictor(
            n, predictor_name, rng
        )

        true_class, indicators = (
            simulate_resources(
                n, k, rng
            )
        )

        evaluation = evaluate_lca_models(
            indicators, rng
        )

        lca = evaluation["models"][k]

        selected_aic = evaluation[
            "selected_aic"
        ]

        selected_bic = evaluation[
            "selected_bic"
        ]

        estimated_class, posterior = (
            align_classes(
                lca["theta"],
                ITEM_PROBABILITIES[k],
                lca["posterior"]
            )
        )

        accuracy = np.mean(
            estimated_class == true_class
        )

        avg_posterior = np.mean(
            np.max(posterior, axis=1)
        )

        class_sizes = np.bincount(
            estimated_class,
            minlength=k
        )

        minimum_proportion = (
            class_sizes.min() / n
        )

        result.update({
            "lca_converged": lca["converged"],
            "lca_aic": lca["aic"],
            "lca_bic": lca["bic"],
            "lca_entropy": lca["entropy"],
            "lca_loglik": lca["loglik"],
            "lca_iterations": lca["iterations"],
            "classification_accuracy": accuracy,
            "average_posterior_probability": avg_posterior,
            "minimum_class_proportion": minimum_proportion,
            "selected_classes_aic": selected_aic,
            "selected_classes_bic": selected_bic,
            "aic_correct": (
                selected_aic == k
            ),
            "bic_correct": (
                selected_bic == k
            ),
            "class_size_acceptable": (
                minimum_proportion >=
                MIN_CLASS_PROPORTION
            )
        })

        y = generate_outcome(
            df,
            x,
            true_class,
            k,
            beta,
            outcome_type,
            rng
        )

        df["predictor"] = x
        df["outcome"] = y

        df["estimated_class"] = pd.Categorical(
            estimated_class,
            categories=list(range(k))
        )

        if np.any(class_sizes == 0):

            result["error"] = (
                "One or more estimated classes empty"
            )

            return result

        lme = fit_lme(
            df,
            outcome_type,
            k
        )

        result.update(lme)

    except Exception as error:

        result["error"] = str(error)[:200]

    return result


# ============================================================
# 13. DEFINE SIMULATION JOBS
# ============================================================

SCENARIOS = list(
    product(
        OUTCOMES,
        PREDICTORS,
        GENERATING_CLASSES,
        EFFECT_SIZES
    )
)

assert len(SCENARIOS) == 120

TOTAL_REPLICATES = (
    len(SCENARIOS) * N_SIMULATIONS
)

seed_sequence = np.random.SeedSequence(
    SEED
)

seeds = seed_sequence.spawn(
    TOTAL_REPLICATES
)

JOBS = []

index = 0

for scenario in SCENARIOS:

    for replicate in range(
        1,
        N_SIMULATIONS + 1
    ):

        JOBS.append((
            scenario,
            replicate,
            seeds[index]
        ))

        index += 1


# ============================================================
# 14. PARALLEL PROCESSING WITH TEXT PROGRESS BAR
# ============================================================

DETAIL_FILE = OUTPUT_DIR / "fit_diagnostics.csv"

if DETAIL_FILE.exists():

    existing = pd.read_csv(
        DETAIL_FILE
    )

    completed = set(
        zip(
            existing["outcome_type"],
            existing["predictor"],
            existing["n_classes"],
            existing["beta"].round(8),
            existing["replicate"]
        )
    )

else:
    completed = set()


PENDING_JOBS = []

for scenario, replicate, seed in JOBS:

    key = (
        scenario[0],
        scenario[1],
        scenario[2],
        round(scenario[3], 8),
        replicate
    )

    if key not in completed:

        PENDING_JOBS.append((
            scenario,
            replicate,
            seed
        ))


print("\n" + "=" * 60, flush=True)
print("HYPOTHESIS 3 MONTE CARLO POWER ANALYSIS", flush=True)
print("=" * 60, flush=True)

print("Total scenarios:", len(SCENARIOS), flush=True)
print("Total simulations:", TOTAL_REPLICATES, flush=True)
print("Already completed:", len(completed), flush=True)
print("Pending:", len(PENDING_JOBS), flush=True)
print("Parallel workers:", N_JOBS, flush=True)

total_batches = int(
    np.ceil(len(PENDING_JOBS) / BATCH_SIZE)
)

progress_bar = tqdm(
    total=total_batches,
    desc="H3 simulation batches",
    ascii=True,
    dynamic_ncols=False,
    ncols=90,
    mininterval=1,
    leave=True,
    file=sys.stdout
)

with threadpool_limits(limits=1):

    for start in range(
        0,
        len(PENDING_JOBS),
        BATCH_SIZE
    ):

        batch = PENDING_JOBS[
            start:start + BATCH_SIZE
        ]

        batch_results = Parallel(
            n_jobs=N_JOBS,
            backend="loky",
            verbose=0
        )(
            delayed(run_one_replicate)(
                scenario,
                replicate,
                seed
            )
            for scenario, replicate, seed in batch
        )

        batch_df = pd.DataFrame(
            batch_results
        )

        batch_df.to_csv(
            DETAIL_FILE,
            mode="a",
            header=not DETAIL_FILE.exists(),
            index=False
        )

        progress_bar.update(1)

        completed_count = (
            len(completed)
            + min(
                start + len(batch),
                len(PENDING_JOBS)
            )
        )

        tqdm.write(
            f"Completed {completed_count:,} "
            f"of {TOTAL_REPLICATES:,} simulations",
            file=sys.stdout
        )

progress_bar.close()

print(
    "\nAll simulation batches completed.",
    flush=True
)


# ============================================================
# 15. LOAD AND SUMMARIZE RESULTS
# ============================================================

details = pd.read_csv(
    DETAIL_FILE
)

details = details.drop_duplicates(
    subset=[
        "outcome_type",
        "predictor",
        "n_classes",
        "beta",
        "replicate"
    ],
    keep="last"
)

details["significant"] = (
    details["p_value"] < ALPHA
)

details["valid_fit"] = (
    details["p_value"].notna()
)

group_cols = [
    "outcome_type",
    "predictor",
    "n_classes",
    "beta"
]

summary = (
    details
    .groupby(group_cols)
    .agg(
        simulations=("replicate", "count"),
        valid_fits=("valid_fit", "sum"),
        significant_fits=("significant", "sum"),
        lca_convergence_rate=(
            "lca_converged", "mean"
        ),
        lme_convergence_rate=(
            "converged", "mean"
        ),
        mean_entropy=(
            "lca_entropy", "mean"
        ),
        mean_classification_accuracy=(
            "classification_accuracy", "mean"
        )
    )
    .reset_index()
)

summary["failed_fits"] = (
    summary["simulations"]
    - summary["valid_fits"]
)

summary["power"] = (
    summary["significant_fits"]
    / summary["valid_fits"].replace(0, np.nan)
)

summary.to_csv(
    OUTPUT_DIR / "power_summary.csv",
    index=False
)


# ============================================================
# 16. LCA DIAGNOSTIC SUMMARY
# ============================================================

lca_summary = (
    details
    .groupby([
        "n_classes",
        "outcome_type"
    ])
    .agg(
        mean_AIC=(
            "lca_aic", "mean"
        ),
        mean_BIC=(
            "lca_bic", "mean"
        ),
        mean_entropy=(
            "lca_entropy", "mean"
        ),
        mean_posterior_probability=(
            "average_posterior_probability",
            "mean"
        ),
        mean_minimum_class_proportion=(
            "minimum_class_proportion",
            "mean"
        ),
        mean_classification_accuracy=(
            "classification_accuracy",
            "mean"
        ),
        aic_correct_selection_rate=(
            "aic_correct", "mean"
        ),
        bic_correct_selection_rate=(
            "bic_correct", "mean"
        ),
        acceptable_class_size_rate=(
            "class_size_acceptable", "mean"
        ),
        lca_convergence_rate=(
            "lca_converged", "mean"
        ),
        lme_convergence_rate=(
            "converged", "mean"
        )
    )
    .reset_index()
)

lca_summary.to_csv(
    OUTPUT_DIR / "LCA_diagnostic_summary.csv",
    index=False
)

print("\nLCA DIAGNOSTIC SUMMARY")

print(
    lca_summary.round(3).to_string(
        index=False
    )
)


# ============================================================
# 17. EMPIRICAL TYPE I ERROR
# ============================================================

type_I_error = summary[
    np.isclose(summary["beta"], 0)
].copy()

type_I_error = type_I_error.rename(
    columns={
        "power": "type_I_error"
    }
)

type_I_error.to_csv(
    OUTPUT_DIR / "type_I_error.csv",
    index=False
)

print("\nEMPIRICAL TYPE I ERROR")

print(
    type_I_error[
        [
            "n_classes",
            "outcome_type",
            "predictor",
            "type_I_error",
            "valid_fits"
        ]
    ].round(3).to_string(index=False)
)


# ============================================================
# 18. MINIMUM DETECTABLE EFFECT SIZE
# ============================================================

detectable = (
    summary.loc[
        (summary["power"] >= TARGET_POWER)
        & (summary["beta"] > 0)
    ]
    .groupby([
        "n_classes",
        "outcome_type",
        "predictor"
    ])["beta"]
    .min()
    .reset_index()
    .rename(
        columns={
            "beta": "minimum_beta_95_power"
        }
    )
)

all_configs = (
    summary[
        [
            "n_classes",
            "outcome_type",
            "predictor"
        ]
    ]
    .drop_duplicates()
)

detectable = all_configs.merge(
    detectable,
    on=[
        "n_classes",
        "outcome_type",
        "predictor"
    ],
    how="left"
)

detectable.to_csv(
    OUTPUT_DIR / "minimum_detectable_effects.csv",
    index=False
)

print("\nMINIMUM TESTED BETA AT 95% POWER")

print(
    detectable.to_string(index=False)
)


# ============================================================
# 19. PLOT POWER CURVES
# ============================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Helvetica",
        "Arial",
        "DejaVu Sans"
    ],
    "font.size": 12,
    "pdf.fonttype": 42
})

DARK_PURPLE = "#43206E"
LIGHT_PURPLE = "#9863C9"
TARGET_RED = "#EF4760"
GRID_COLOR = "#EEE9F5"

for outcome_type in OUTCOMES:

    for predictor in PREDICTORS:

        subset = summary[
            (summary["outcome_type"] == outcome_type)
            & (summary["predictor"] == predictor)
        ]

        fig, ax = plt.subplots(
            figsize=(10, 6.2)
        )

        for k, color, linestyle, marker in [
            (2, DARK_PURPLE, "-", "o"),
            (3, LIGHT_PURPLE, "--", "s")
        ]:

            data = subset[
                subset["n_classes"] == k
            ].sort_values("beta")

            ax.plot(
                data["beta"],
                data["power"],
                color=color,
                linestyle=linestyle,
                marker=marker,
                linewidth=2.6,
                markersize=6,
                label=f"{k} classes"
            )

        ax.axhline(
            TARGET_POWER,
            color=TARGET_RED,
            linestyle="--",
            linewidth=1.8,
            label="95% target"
        )

        ax.set_xlim(-0.01, 0.51)
        ax.set_ylim(0, 1.03)

        ax.set_xticks(
            np.arange(0, 0.51, 0.05)
        )

        ax.set_yticks(
            np.arange(0, 1.01, 0.2)
        )

        ax.set_xlabel(
            "Assumed interaction coefficients (β)",
            fontsize=14,
            color=DARK_PURPLE,
            labelpad=12
        )

        ax.set_ylabel(
            "Estimated power",
            fontsize=14,
            color=DARK_PURPLE,
            labelpad=12
        )

        ax.set_title(
            f"H3: {outcome_type} — {predictor}",
            fontsize=17,
            color=DARK_PURPLE,
            pad=18
        )

        ax.grid(
            axis="y",
            color=GRID_COLOR,
            linewidth=0.8
        )

        ax.set_axisbelow(True)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.spines["left"].set_color(
            DARK_PURPLE
        )

        ax.spines["bottom"].set_color(
            DARK_PURPLE
        )

        ax.tick_params(
            axis="both",
            colors=DARK_PURPLE
        )

        ax.legend(
            loc="lower right",
            fontsize=11,
            frameon=True,
            facecolor="white",
            edgecolor="#E8DDF0"
        )

        plt.tight_layout()

        safe_name = (
            predictor.lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        filename = (
            OUTPUT_DIR
            / f"H3_power_{outcome_type}_{safe_name}"
        )

        fig.savefig(
            str(filename) + ".png",
            dpi=1000,
            bbox_inches="tight"
        )

        fig.savefig(
            str(filename) + ".pdf",
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)


# ============================================================
# 20. SAVE SIMULATION CONFIGURATION
# ============================================================

configuration = {
    "N_GM": N_GM,
    "N_WM": N_WM,
    "N_STATES": N_STATES,
    "N_SITES": N_SITES,
    "N_SIMULATIONS": N_SIMULATIONS,
    "EFFECT_SIZES": EFFECT_SIZES,
    "ALPHA": ALPHA,
    "TARGET_POWER": TARGET_POWER,
    "CLASS_PROPORTIONS": {
        str(k): v.tolist()
        for k, v in CLASS_PROPORTIONS.items()
    },
    "RESOURCE_PREVALENCE": (
        RESOURCE_PREVALENCE.tolist()
    ),
    "ITEM_PROBABILITIES": {
        str(k): v.tolist()
        for k, v in ITEM_PROBABILITIES.items()
    }
}

with open(
    OUTPUT_DIR / "simulation_config.json",
    "w"
) as file:

    json.dump(
        configuration,
        file,
        indent=4
    )


# ============================================================
# 21. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("HYPOTHESIS 3 SIMULATION COMPLETED")
print("=" * 60)

print(
    "Total simulations:",
    len(details)
)

print(
    "Valid LME fits:",
    details["valid_fit"].sum()
)

print(
    "Failed LME fits:",
    (~details["valid_fit"]).sum()
)

print(
    "Mean LCA entropy:",
    round(
        details["lca_entropy"].mean(),
        3
    )
)

print(
    "Mean classification accuracy:",
    round(
        details["classification_accuracy"].mean(),
        3
    )
)

print(
    "Mean AIC correct selection rate:",
    round(
        details["aic_correct"].mean(),
        3
    )
)

print(
    "Mean BIC correct selection rate:",
    round(
        details["bic_correct"].mean(),
        3
    )
)

print(
    "\nResults saved to:",
    OUTPUT_DIR.resolve()
)
